In [0]:
display(dbutils.fs.ls("/Volumes/retail_q/crm_bronze/crm_source/"))

In [0]:
accounts_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/retail_q/crm_bronze/crm_source/accounts.csv")
)

display(accounts_df)

In [0]:
# CRM Bronze - Accounts
accounts_df.write.format("delta").mode("overwrite").saveAsTable(
    "retail_q.crm_bronze.accounts"
)

# CRM Bronze - Sales Pipeline / Opportunities
sales_pipeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/retail_q/crm_bronze/crm_source/sales_pipeline.csv")
)

sales_pipeline_df.write.format("delta").mode("overwrite").saveAsTable(
    "retail_q.crm_bronze.sales_pipeline"
)

# CRM Bronze - Products
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/retail_q/crm_bronze/crm_source/products.csv")
)

products_df.write.format("delta").mode("overwrite").saveAsTable(
    "retail_q.crm_bronze.products"
)

# CRM Bronze - Sales Teams
sales_teams_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/retail_q/crm_bronze/crm_source/sales_teams.csv")
)

sales_teams_df.write.format("delta").mode("overwrite").saveAsTable(
    "retail_q.crm_bronze.sales_teams"
)

print("All 4 CRM Bronze tables created successfully!")

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CREATE SILVER SCHEMA
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS retail_q.crm_silver")


# ============================================================
# 1. ACCOUNT → SILVER
# ============================================================

accounts_silver = (
    spark.table("retail_q.crm_bronze.accounts")
    .select(
        F.trim(F.col("account")).alias("account"),
        F.trim(F.col("sector")).alias("sector"),
        F.col("year_established").cast("int").alias("year_established"),
        F.col("revenue").cast("double").alias("revenue"),
        F.col("employees").cast("int").alias("employees"),
        F.trim(F.col("office_location")).alias("office_location"),
        F.trim(F.col("subsidiary_of")).alias("subsidiary_of")
    )
    .withColumn(
        "account",
        F.when(F.col("account") == "", None).otherwise(F.col("account"))
    )
    .withColumn("processed_at", F.current_timestamp())
)

accounts_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.crm_silver.dim_account")


# ============================================================
# 2. PRODUCTS → SILVER
# ============================================================

products_silver = (
    spark.table("retail_q.crm_bronze.products")
    .select(
        F.trim(F.col("product")).alias("product"),
        F.trim(F.col("series")).alias("series"),
        F.col("sales_price").cast("double").alias("sales_price")
    )
    .withColumn(
        "product",
        F.when(F.col("product") == "", None).otherwise(F.col("product"))
    )
    .withColumn(
        "product_segment",
        F.when(F.col("series").isNull(), "Unknown")
         .otherwise(F.col("series"))
    )
    .withColumn("processed_at", F.current_timestamp())
)

products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.crm_silver.dim_product")


# ============================================================
# 3. SALES TEAM → SILVER
# ============================================================

sales_team_silver = (
    spark.table("retail_q.crm_bronze.sales_teams")
    .select(
        F.trim(F.col("sales_agent")).alias("sales_agent"),
        F.trim(F.col("manager")).alias("manager"),
        F.trim(F.col("regional_office")).alias("regional_office")
    )
    .withColumn("processed_at", F.current_timestamp())
)

sales_team_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.crm_silver.dim_sales_team")


# ============================================================
# 4. SALES PIPELINE / OPPORTUNITY → SILVER
# ============================================================

opportunity_silver = (
    spark.table("retail_q.crm_bronze.sales_pipeline")
    .select(
        F.trim(F.col("opportunity_id")).alias("opportunity_id"),
        F.trim(F.col("sales_agent")).alias("sales_agent"),
        F.trim(F.col("product")).alias("product"),
        F.trim(F.col("account")).alias("account"),
        F.trim(F.col("deal_stage")).alias("deal_stage"),

        F.to_date(
            F.col("engage_date"),
            "yyyy-MM-dd"
        ).alias("engage_date"),

        F.to_date(
            F.col("close_date"),
            "yyyy-MM-dd"
        ).alias("close_date"),

        F.col("close_value").cast("double").alias("close_value")
    )
    .withColumn(
        "opportunity_status",
        F.when(
            F.lower(F.col("deal_stage")).isin(
                "won",
                "closed won"
            ),
            "Won"
        )
        .when(
            F.lower(F.col("deal_stage")).isin(
                "lost",
                "closed lost"
            ),
            "Lost"
        )
        .otherwise("Open")
    )
    .withColumn(
        "sales_cycle_days",
        F.when(
            F.col("close_date").isNotNull() &
            F.col("engage_date").isNotNull(),
            F.datediff(
                F.col("close_date"),
                F.col("engage_date")
            )
        )
    )
    .withColumn("processed_at", F.current_timestamp())
)

opportunity_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.crm_silver.fact_opportunity")


# ============================================================
# 5. BASIC DATA QUALITY CHECKS
# ============================================================

print("========== SILVER TABLE COUNTS ==========")

print(
    "Accounts:",
    spark.table("retail_q.crm_silver.dim_account").count()
)

print(
    "Products:",
    spark.table("retail_q.crm_silver.dim_product").count()
)

print(
    "Sales Teams:",
    spark.table("retail_q.crm_silver.dim_sales_team").count()
)

print(
    "Opportunities:",
    spark.table("retail_q.crm_silver.fact_opportunity").count()
)

print("=========================================")
print("✅ CRM SILVER TRANSFORMATION COMPLETED")

In [0]:
spark.sql("SHOW TABLES IN retail_q.blob_bronze").show(truncate=False)

In [0]:
spark.sql("SHOW SCHEMAS IN retail_q").show(truncate=False)

In [0]:
spark.sql("SHOW TABLES IN retail_q.postgress_to_bronze").show(truncate=False)

In [0]:
display(dbutils.fs.ls("/Volumes/retail_q/"))

In [0]:
display(dbutils.fs.ls("/Volumes/retail_q/postgress_to_bronze/"))

In [0]:
display(dbutils.fs.ls("/Volumes/retail_q/crm_bronze/crm_source/"))

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
schemas = [
    r.databaseName
    for r in spark.sql("SHOW SCHEMAS IN retail_q").collect()
    if r.databaseName not in ["information_schema"]
]

for schema in schemas:
    print(f"\n===== {schema} =====")
    spark.sql(f"SHOW TABLES IN retail_q.`{schema}`").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CREATE GOLD SCHEMA
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS retail_q.retail_gold")


# ============================================================
# 1. CUSTOMER DIMENSION
# ============================================================

dim_customer = (
    spark.table("retail_q.crm_silver.dim_account")
    .select(
        "account",
        "sector",
        "year_established",
        "revenue",
        "employees",
        "office_location",
        "subsidiary_of"
    )
    .withColumnRenamed("account", "customer_name")
    .withColumnRenamed("sector", "customer_industry")
    .withColumnRenamed("office_location", "billing_location")
)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.dim_customer")


# ============================================================
# 2. PRODUCT DIMENSION
# ============================================================

dim_product = (
    spark.table("retail_q.crm_silver.dim_product")
    .select(
        "product",
        "series",
        "sales_price",
        "product_segment"
    )
    .withColumnRenamed("product", "product_name")
)

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.dim_product")


# ============================================================
# 3. SALES TEAM DIMENSION
# ============================================================

dim_sales_team = (
    spark.table("retail_q.crm_silver.dim_sales_team")
    .select(
        "sales_agent",
        "manager",
        "regional_office"
    )
)

dim_sales_team.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.dim_sales_team")


# ============================================================
# 4. FACT SALES
#    WON OPPORTUNITIES BECOME SALES
# ============================================================

fact_sales = (
    spark.table("retail_q.crm_silver.fact_opportunity")
    .filter(F.col("opportunity_status") == "Won")
    .select(
        "opportunity_id",
        "account",
        "product",
        "sales_agent",
        "deal_stage",
        "engage_date",
        "close_date",
        "close_value",
        "sales_cycle_days"
    )
    .withColumnRenamed("account", "customer_name")
    .withColumnRenamed("product", "product_name")
    .withColumnRenamed("close_value", "sales_amount")
    .withColumn(
        "quantity_sold",
        F.lit(1)
    )
    .withColumn(
        "transaction_month",
        F.date_format("close_date", "yyyy-MM")
    )
    .withColumn(
        "transaction_year",
        F.year("close_date")
    )
    .withColumn(
        "transaction_quarter",
        F.quarter("close_date")
    )
    .withColumn("processed_at", F.current_timestamp())
)

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.fact_sales")


# ============================================================
# 5. INVENTORY GOLD
# ============================================================

fact_inventory = (
    spark.table("retail_q.postgress_to_bronze.inventory")
    .select(
        "inventory_id",
        "product_id",
        "store_id",
        "stock_quantity",
        "reorder_level",
        "warehouse_location",
        "last_stock_update"
    )
    .withColumn(
        "inventory_status",
        F.when(
            F.col("stock_quantity") <= F.col("reorder_level"),
            "Reorder Required"
        )
        .otherwise("Healthy")
    )
)

fact_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.fact_inventory")


# ============================================================
# 6. VALIDATION
# ============================================================

print("========== GOLD TABLE COUNTS ==========")

for table_name in [
    "dim_customer",
    "dim_product",
    "dim_sales_team",
    "fact_sales",
    "fact_inventory"
]:
    count = spark.table(
        f"retail_q.retail_gold.{table_name}"
    ).count()

    print(f"{table_name}: {count}")

print("========================================")
print("✅ GOLD LAYER COMPLETED")

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CREATE CALENDAR DIMENSION
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS retail_q.retail_gold")

min_date, max_date = (
    spark.table("retail_q.retail_gold.fact_sales")
    .select(
        F.min("close_date").alias("min_date"),
        F.max("close_date").alias("max_date")
    )
    .first()
)

calendar_df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS calendar_date
    """)
    .withColumn("year", F.year("calendar_date"))
    .withColumn("quarter", F.quarter("calendar_date"))
    .withColumn("month", F.month("calendar_date"))
    .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
    .withColumn("week", F.weekofyear("calendar_date"))
    .withColumn("day", F.dayofmonth("calendar_date"))
    .withColumn("day_name", F.date_format("calendar_date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("calendar_date").isin([1, 7])
    )
    .withColumn(
        "year_month",
        F.date_format("calendar_date", "yyyy-MM")
    )
    .withColumn(
        "first_day_of_month",
        F.trunc("calendar_date", "month")
    )
    .withColumn(
        "last_day_of_month",
        F.last_day("calendar_date")
    )
)

calendar_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_q.retail_gold.dim_calendar")

print("Calendar rows:", calendar_df.count())
print("✅ Calendar dimension created")


# ============================================================
# CREATE SEMANTIC SCHEMA
# ============================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS retail_q.retail_semantics"
)


# ============================================================
# CREATE SEMANTIC BASE TABLE
# ============================================================

fact = spark.table("retail_q.retail_gold.fact_sales").alias("f")
customer = spark.table("retail_q.retail_gold.dim_customer").alias("c")
product = spark.table("retail_q.retail_gold.dim_product").alias("p")
team = spark.table("retail_q.retail_gold.dim_sales_team").alias("t")
calendar = spark.table("retail_q.retail_gold.dim_calendar").alias("cal")

semantic_df = (
    fact
    .join(
        customer,
        F.col("f.customer_name") ==
        F.col("c.customer_name"),
        "left"
    )
    .join(
        product,
        F.col("f.product_name") ==
        F.col("p.product_name"),
        "left"
    )
    .join(
        team,
        F.col("f.sales_agent") ==
        F.col("t.sales_agent"),
        "left"
    )
    .join(
        calendar,
        F.col("f.close_date") ==
        F.col("cal.calendar_date"),
        "left"
    )
    .select(
        F.col("f.opportunity_id"),
        F.col("f.sales_amount"),
        F.col("f.quantity_sold"),
        F.col("f.deal_stage"),
        F.col("f.engage_date"),
        F.col("f.close_date"),
        F.col("f.sales_cycle_days"),

        F.col("c.customer_name"),
        F.col("c.customer_industry"),
        F.col("c.revenue").alias("customer_revenue"),
        F.col("c.employees").alias("customer_employees"),
        F.col("c.billing_location"),

        F.col("p.product_name"),
        F.col("p.series"),
        F.col("p.sales_price"),
        F.col("p.product_segment"),

        F.col("t.sales_agent"),
        F.col("t.manager"),
        F.col("t.regional_office"),

        F.col("cal.year").alias("transaction_year"),
        F.col("cal.quarter").alias("transaction_quarter"),
        F.col("cal.month").alias("transaction_month_number"),
        F.col("cal.month_name").alias("transaction_month"),
        F.col("cal.year_month"),
        F.col("cal.is_weekend")
    )
)

semantic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "retail_q.retail_semantics.retail_metrics"
    )


# ============================================================
# VALIDATION
# ============================================================

print(
    "Semantic rows:",
    spark.table(
        "retail_q.retail_semantics.retail_metrics"
    ).count()
)

print("✅ SEMANTIC LAYER COMPLETED")